In [36]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
import IPython.display as ipd
from scipy import signal


def gen_noise(Ln,cutoff):    
    Tdur = 3 # noise duration (seconds)
    fs = 44100 # sample rate (Hz)
    dt = 1/fs  # time step
    tx = np.arange(0,Tdur,dt)  # time axis
    nsamp = len(tx)
    amp = 10**(Ln/20) # amplitude of the noise
    noise = amp*np.random.normal(0,1,nsamp) # broad band noise

    # make a low pass filter up to 10 kHz to make the noise less annoying 
    
    normalCutoff = cutoff / (fs/2)
    order = 15
    bLP, aLP = signal.butter(order, normalCutoff, btype='low')

    fnoise = signal.lfilter(bLP, aLP, noise)

    return fnoise, tx
#ipd.display(ipd.Audio(hpitch, rate=fs))


#fig, ax = plt.subplots(figsize=(13, 6))
#ax.set_xlabel('time')
#ax.set_ylabel('amplitude')
     
#ax.plot(tx, amp*noise, lw=3, c='r')
#tone = gen_tone(amp,1e3,tx,0)
#ax.plot(tx, tone, lw=3, c='r')

def gen_tone(amp,freq,tx,phase):
    '''generate tone '''
    
    tone = amp*np.sin(2*np.pi*freq*tx + phase)
    # make a ramp (fade in fade out)
    Rdur = 50e-3  # ramp duration
    x = np.arange(0,Rdur,tx[1]-tx[0])
    x = np.pi*x/Rdur
    rampUp = (1 + np.cos(x + np.pi))/2; # raised cosine onset
    rampDown = np.flip(rampUp)
    
    wholeramp = np.concatenate((rampUp, np.ones(len(tone)-2*len(x)), rampDown))
    
    tone = wholeramp*tone

        
    return tone

def update_signal(Lt):
    global fnoise, tx
    ampt = 10**(Lt/20)
    freq = 1e3
    tone_l = gen_tone(ampt,freq,tx,0)
    tone_r = gen_tone(ampt,freq,tx,0)
    signal_l = tone_l + fnoise
    signal_r = tone_r + fnoise
    signal = [signal_l, signal_r]  # pitch sensation
    signal_tone = [tone_l, tone_r]
    #fig, ax = plt.subplots(figsize=(13, 6))
    #ax.set_xlabel('time (seconds)')
    #ax.set_ylabel('Amplitude')
     
    #ax.plot(tx,signal, lw=3, c='r')
    fs = 44100
    display(ipd.Audio(signal, rate=fs, autoplay=True,normalize=False))
    display(ipd.Audio(signal_tone, rate=fs, autoplay=True,normalize=False))
    #ipd.display(ipd.Audio(signal, rate=fs,normalize=False))

global fnoise, tx

fnoise,tx = gen_noise(-20,2e3)


#c_slide = widgets.IntSlider(min=0,max=180,step=10,description='phase diff')
s_slide = widgets.IntSlider(min=-50,max=-20,step=1,description='tone level')

widgets.interact(update_signal, Lt=s_slide)








interactive(children=(IntSlider(value=-20, description='tone level', max=-20, min=-50), Output()), _dom_classe…

<function __main__.update_signal(Lt)>

In [35]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
import IPython.display as ipd
from IPython.display import display

# parameters
fs = 44100
Tdur = 2
t = np.arange(0, Tdur, 1/fs)

freq = 1000  # tone frequency
noise_amp = 0.05  # amplitude of added noise

# generate tone
def gen_tone(freq=1000, amp=0.1):
    tone = amp * np.sin(2*np.pi*freq*t)
    
    # fade in/out to avoid clicks
    Rdur = 50e-3
    n = int(Rdur * fs)
    ramp = np.linspace(0, 1, n)
    tone[:n] *= ramp
    tone[-n:] *= ramp[::-1]
    
    return tone

# quantization
def quantize_signal(x, bits):
    levels = 2**bits
    xq = np.round((x + 1)/2 * (levels - 1)) / (levels - 1)
    xq = 2*xq - 1
    return xq

# main function
def play_and_plot(bits, noise=False):
    tone = gen_tone(freq=freq)
    
    # quantize first
    qtone = quantize_signal(tone, bits)
    
    # add noise after quantization if selected
    if noise:
        qtone += noise_amp * np.random.randn(len(qtone))
        qtone = np.clip(qtone, -1, 1)
    
    # --- AUDIO ---
    display(ipd.Audio(qtone, rate=fs, autoplay=True, normalize=False))
    
    # --- PLOT (few periods) ---
    periods_to_show = 5
    samples_per_period = int(fs / freq)
    N = periods_to_show * samples_per_period
    idxS = 10000
    tt = t[idxS:idxS+N]
    qtone_seg = qtone[idxS:idxS+N]
    
    plt.figure(figsize=(10,4))
    plt.step(tt*1000, qtone_seg, where='mid', label='quantized' + (' + noise' if noise else ''))
    
    plt.xlabel('Time (ms)')
    plt.ylabel('Amplitude')
    plt.title(f'Quantization with {bits} bits')
    plt.legend()
    plt.grid(True)
    plt.show()

# widgets
bit_slider = widgets.IntSlider(
    value=8,
    min=5,
    max=16,
    step=1,
    description='bits'
)

noise_toggle = widgets.ToggleButtons(
    options=[('Clean Quantized', False), ('Quantized + Noise', True)],
    description='Audio Type:',
    button_style=''
)

widgets.interact(play_and_plot, bits=bit_slider, noise=noise_toggle)

interactive(children=(IntSlider(value=8, description='bits', max=16, min=5), ToggleButtons(description='Audio …

<function __main__.play_and_plot(bits, noise=False)>

In [31]:
import numpy as np
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, Audio

# Parameters
fs = 44100
Tdur = 2
amp = 0.05
f_fixed = 1000  # Fixed tone

# Generate ideal spectrum (simulated)
def plot_ideal_spectrum(f_var):
    # Audio for demonstration
    t = np.arange(0, Tdur, 1/fs)
    tone1 = amp * np.sin(2*np.pi*f_fixed*t)
    tone2 = amp * np.sin(2*np.pi*f_var*t)
    signal = tone1 + tone2
    
    display(Audio(signal, rate=fs, autoplay=True, normalize=False))
    
    # --- Simulated amplitude spectrum ---
    f_axis = np.linspace(800, 1200, 5000)  # zoomed near 1 kHz
    spectrum = np.zeros_like(f_axis)
    
    # Represent each tone as a narrow Gaussian peak for visualization
    sigma = 0.2  # width of peak (Hz)
    spectrum += amp * np.exp(-0.5*((f_axis - f_fixed)/sigma)**2)
    spectrum += amp * np.exp(-0.5*((f_axis - f_var)/sigma)**2)
    
    plt.figure(figsize=(10,4))
    plt.plot(f_axis, spectrum, lw=2)
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('Amplitude')
    plt.title(f'Ideal Spectrum: {f_fixed} Hz + {f_var:.1f} Hz')
    plt.grid(True)
    plt.show()

# Widget
f_slider = widgets.FloatSlider(
    value=1005,
    min=800,
    max=1200,
    step=1,
    description='Variable freq (Hz)',
    readout_format='.1f'
)

widgets.interact(plot_ideal_spectrum, f_var=f_slider)

interactive(children=(FloatSlider(value=1005.0, description='Variable freq (Hz)', max=1200.0, min=800.0, reado…

<function __main__.plot_ideal_spectrum(f_var)>